# House Price Prediction — Internship Capstone Project

**Project:** Predict residential property sale prices using the Ames Housing dataset  
**Intern submission:** Single Jupyter notebook covering EDA, modeling, comparison, and challenges

**Problem tasks (from internship brief):**
1. Prepare a complete data analysis report on the given data
2. Create predictive models for house prices using available independent variables
3. Produce a model comparison report and recommend a production model
4. Document challenges faced and techniques used (with reasons)

**Dataset:** `data.csv` (Ames housing-style data; target = `SalePrice`)  
**Reference docs:** `house prices.docx` (field dictionary), `PRCP-1017-AutoPricePred (4).docx` (task structure)

**This revision emphasizes reviewer feedback themes:**
- EDA observations explicitly tied to preprocessing / modeling decisions
- Justified preprocessing (median vs mean, categorical encoding, scaling)
- GridSearchCV hyperparameter tuning + model interpretability
- Specific, quantified future-work suggestions in the conclusion


## 0. Environment setup

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries loaded successfully.")
print("XGBoost available:", HAS_XGB)


## 1. Load data & first look

In [ ]:
df = pd.read_csv("data.csv")
print("Shape:", df.shape)
print("Rows = houses, Columns = features + target (SalePrice)")
df.head()

In [ ]:
df.info()

In [ ]:
# Basic statistics for numeric columns
df.describe().T

In [ ]:
# Target check
assert "SalePrice" in df.columns, "SalePrice column missing!"
print("Target variable: SalePrice")
print(df["SalePrice"].describe())

## 2. Task 1 — Complete Data Analysis Report

### 2.1 Missing values overview
In this dataset, many `NaN` values are **not random** — they mean a feature is absent
(e.g. no pool → `PoolQC` missing, no garage → garage fields missing). That changes how we clean the data.

**Modeling implication:** Treating amenity-related missingness as a real category (`"None"` / `0`) preserves signal.
Dropping those columns or filling them with the mode would invent amenities that do not exist and can hurt price predictions.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)
missing_df = missing_df[missing_df["missing_count"] > 0]
print(f"Columns with missing values: {len(missing_df)}")
missing_df

In [ ]:
plt.figure(figsize=(12, 6))
top_missing = missing_df.head(15)
sns.barplot(x=top_missing["missing_pct"], y=top_missing.index, color="#2a6f97")
plt.title("Top 15 Columns by Missing Value Percentage")
plt.xlabel("Missing %")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

**Observation → preprocessing link**
- Columns with >80% missing (`PoolQC`, `MiscFeature`, `Alley`, `Fence`) still matter: presence/absence of a pool or alley can shift price.
- `FireplaceQu` and garage/basement quality fields are missing together with related numeric fields → impute as a block (`None` + `0`).
- `LotFrontage` (~17–18% missing) looks like true missing data, not “feature absent” → use **median** imputation (robust to skewed lot sizes), preferably by neighborhood.


### 2.2 Target distribution (`SalePrice`)
House prices are usually right-skewed (many typical homes + a few expensive outliers).
Models often work better on a log-transformed target.

**Modeling implication:** High skew inflates squared-error loss on expensive homes and hurts linear models.
Training on `log1p(SalePrice)` and converting predictions back with `expm1` typically reduces skew toward ~0 and improves RMSLE — a fairer relative-error metric for pricing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0], color="#2a6f97")
axes[0].set_title("SalePrice Distribution (Raw)")
axes[0].set_xlabel("SalePrice ($)")

sns.histplot(np.log1p(df["SalePrice"]), kde=True, ax=axes[1], color="#01497c")
axes[1].set_title("SalePrice Distribution (Log1p)")
axes[1].set_xlabel("log(1 + SalePrice)")

plt.tight_layout()
plt.show()

print("Skewness (raw):", round(df["SalePrice"].skew(), 3))
print("Skewness (log1p):", round(np.log1p(df["SalePrice"]).skew(), 3))

**Observation → modeling link**
- Raw `SalePrice` skew is typically >1 (right-skewed); after `log1p` it usually drops near 0.
- **Implication:** Use a log target for all models in this notebook. Evaluate RMSE/MAE on the original dollar scale after inverse transform so results stay business-readable.


### 2.3 Numeric feature correlations with SalePrice

Strong linear associations hint at useful predictors, but **multicollinearity** among size-related features
(`GrLivArea`, `1stFlrSF`, `TotalBsmtSF`, garage size) can destabilize ordinary least squares.

**Modeling implication:** Prefer regularized linear models (Ridge / Lasso / ElasticNet) and/or engineer condensed features such as `TotalSF`. Tree ensembles handle collinearity more gracefully but still benefit from cleaner features.


In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).copy()
corr = numeric_df.corr(numeric_only=True)
target_corr = corr["SalePrice"].drop("SalePrice").sort_values(ascending=False)

print("Top positive correlations with SalePrice:")
print(target_corr.head(12))
print("\nTop negative correlations with SalePrice:")
print(target_corr.tail(8))

In [ ]:
# Heatmap of top correlated numeric features
top_feats = target_corr.abs().sort_values(ascending=False).head(12).index.tolist()
heat_cols = top_feats + ["SalePrice"]
plt.figure(figsize=(11, 8))
sns.heatmap(df[heat_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="Blues", square=True)
plt.title("Correlation Heatmap — Top Features vs SalePrice")
plt.tight_layout()
plt.show()

**Observation → preprocessing / modeling link**
- Top correlates (`OverallQual`, `GrLivArea`, `GarageCars`, `GarageArea`, `TotalBsmtSF`, `1stFlrSF`) should be prioritized in feature engineering and interpretability plots.
- Negative correlates like `HouseAge`-related fields (after engineering) will help explain discounts on older homes.
- Because several size features move together, we will add `TotalSF` / `TotalBath` and rely on regularization for linear baselines.


### 2.4 Key relationship plots
Quality, living area, and garage capacity are typically strong price drivers.

**Modeling implication:** `OverallQual` shows a clear stepwise (almost monotonic) lift in price — ordinal signal that both linear and tree models can use.
`GrLivArea` is roughly linear but fans out at higher values (heteroscedasticity), which again supports a log-price target. A few large-area / low-price points will be treated as outliers before fitting.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", ax=axes[0, 0], alpha=0.6, color="#2a6f97")
axes[0, 0].set_title("Living Area vs SalePrice")

sns.boxplot(data=df, x="OverallQual", y="SalePrice", ax=axes[0, 1], color="#61a5c2")
axes[0, 1].set_title("Overall Quality vs SalePrice")

sns.boxplot(data=df, x="GarageCars", y="SalePrice", ax=axes[1, 0], color="#89c2d9")
axes[1, 0].set_title("Garage Cars vs SalePrice")

sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice", ax=axes[1, 1], alpha=0.6, color="#01497c")
axes[1, 1].set_title("Basement Area vs SalePrice")

plt.tight_layout()
plt.show()

**Observation → modeling link**
- Price rises sharply with `OverallQual` (especially 8–10) → quality upgrades are a concrete lever for management.
- `GarageCars` plateaus around 3 cars; rare 4-car homes may be noisy — trees can isolate them, linear models may over-extrapolate.
- Basement and living area both matter; combining them into `TotalSF` reduces redundancy for linear models.


### 2.5 Categorical insights — Neighborhood & Zoning

Location is one of the strongest non-numeric drivers. Average prices differ widely across neighborhoods,
and some neighborhoods have relatively few sales — a **data imbalance / sparse-category** challenge for modeling
(analogous to rare classes in classification problems).

**Modeling implication:** Keep `Neighborhood` and `MSZoning` as encoded categoricals. Rare neighborhoods can cause unstable estimates;
tree models and regularization help, and future work could use target encoding or neighborhood grouping.


In [ ]:
# Average sale price by neighborhood
nbhd = df.groupby("Neighborhood")["SalePrice"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=nbhd.index, y=nbhd.values, color="#2a6f97")
plt.xticks(rotation=75)
plt.title("Average SalePrice by Neighborhood")
plt.ylabel("Average SalePrice ($)")
plt.xlabel("Neighborhood")
plt.tight_layout()
plt.show()

print("Most expensive neighborhoods (avg):")
print(nbhd.head(5))
print("\nLeast expensive neighborhoods (avg):")
print(nbhd.tail(5))

In [ ]:
# Zoning impact
zone = df.groupby("MSZoning")["SalePrice"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(zone)

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="MSZoning", y="SalePrice", color="#61a5c2")
plt.title("SalePrice by Zoning Classification (MSZoning)")
plt.tight_layout()
plt.show()

**Observation → preprocessing / modeling link**
- Premium areas (e.g. NoRidge, NridgHt, StoneBr) vs lower-priced areas (e.g. MeadowV, IDOTRR) create large location effects — encoding these categories is essential, not optional.
- Low-count neighborhoods are a modeling risk: predictions there will have higher variance. Stratified sampling is not directly applicable to continuous prices, but a fixed random split with a held-out test set still gives an honest check; future work could bin prices and stratify.
- Zoning (`MSZoning`) also shifts the price distribution → must be encoded before modeling.


### 2.6 Outlier check (GrLivArea)
Very large living areas with unusually low prices can distort linear models. We flag them here and remove extreme outliers before modeling.

**Modeling implication:** Even 2–4 extreme points can pull linear slopes and inflate RMSE. Tree models are more robust, but removing clear partial-sale / anomaly points improves all models and makes metrics more comparable.


In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", alpha=0.6, color="#2a6f97")
plt.axvline(4000, color="red", linestyle="--", label="GrLivArea = 4000")
plt.title("Potential Outliers: Large Homes with Unusual Prices")
plt.legend()
plt.tight_layout()
plt.show()

outliers = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)]
print("Large living area but relatively low price outliers:")
outliers[["Id", "GrLivArea", "OverallQual", "SalePrice"]]

### 2.7 EDA summary (Task 1 findings) — linked to preprocessing & modeling

| EDA finding | Preprocessing / modeling implication |
|-------------|--------------------------------------|
| `SalePrice` is right-skewed | Train on `log1p(SalePrice)`; evaluate on dollars after `expm1` |
| Strong predictors: `OverallQual`, `GrLivArea`, garage & basement size, `YearBuilt` | Keep these features; build `TotalSF`, `TotalBath`, `HouseAge` |
| Neighborhood / zoning drive large price gaps; some areas have few sales | Encode categoricals; treat rare neighborhoods as a sparsity risk for linear models |
| Amenity columns mostly “missing” because amenity absent | Impute with `"None"` / `0` — do **not** use mode/mean that invents amenities |
| `LotFrontage` truly missing (~18%) and skewed | **Median** (neighborhood-wise) beats mean — mean is pulled by large lots |
| Size features are collinear | Regularized linear models + engineered totals; trees as nonlinear alternative |
| A few huge `GrLivArea` homes with low price | Remove before training to protect linear fits and stabilize RMSE |

**Key modeling challenge to highlight:** sparse / low-count neighborhoods behave like rare classes — they make average error look fine overall while local predictions remain unreliable. That motivates cross-validation, regularization, and (later) more data collection or target encoding.


## 3. Data cleaning & feature engineering

We prepare a modeling copy of the data with explicit, internship-style cleaning steps and documented reasons.

**Why these choices (tied to EDA):**
1. **Drop extreme GrLivArea outliers** — protects linear models from a handful of anomalous sales.
2. **`"None"` / `0` for amenity NA** — missingness means absence, which is informative.
3. **Median for `LotFrontage`** — lot frontage is right-skewed; the mean would over-impute large values. Neighborhood median captures location norms better than a global constant.
4. **Mode for `Electrical`** — only ~1 missing row; mode is appropriate for a nearly-complete categorical.
5. **Engineered size / age / amenity flags** — compress collinear signals and expose nonlinear presence effects (`HasGarage`, `HasPool`, …).


In [ ]:
# Work on a clean copy
data = df.copy()

# Remove extreme outliers observed in EDA (large area, low price)
before = len(data)
data = data[~((data["GrLivArea"] > 4000) & (data["SalePrice"] < 300000))]
print(f"Removed {before - len(data)} outlier rows. New shape: {data.shape}")

# Drop Id — identifier only, not predictive
if "Id" in data.columns:
    data = data.drop(columns=["Id"])

In [ ]:
# Meaningful "None" for features where NA means feature absent (EDA: not MCAR)
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]
for col in none_cols:
    if col in data.columns:
        data[col] = data[col].fillna("None")

# Numeric amenity-related zeros (no basement / no masonry / no garage year → 0)
zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]
for col in zero_cols:
    if col in data.columns:
        data[col] = data[col].fillna(0)

# LotFrontage: MEDIAN (not mean) — frontage is skewed; mean would bias toward large lots.
# Neighborhood-wise median first (location-aware), then overall median fallback.
if "LotFrontage" in data.columns:
    data["LotFrontage"] = data.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )
    data["LotFrontage"] = data["LotFrontage"].fillna(data["LotFrontage"].median())

# Electrical: only 1 missing — mode is justified for a near-complete categorical
if "Electrical" in data.columns:
    data["Electrical"] = data["Electrical"].fillna(data["Electrical"].mode()[0])

print("Remaining missing values:", int(data.isnull().sum().sum()))
print("Imputation strategy: None/0 for absent amenities | median for LotFrontage | mode for Electrical")


In [ ]:
# Feature engineering — create useful derived features
data["TotalSF"] = data["TotalBsmtSF"] + data["1stFlrSF"] + data["2ndFlrSF"]
data["TotalBath"] = (
    data["FullBath"]
    + 0.5 * data["HalfBath"]
    + data["BsmtFullBath"]
    + 0.5 * data["BsmtHalfBath"]
)
data["HouseAge"] = data["YrSold"] - data["YearBuilt"]
data["RemodAge"] = data["YrSold"] - data["YearRemodAdd"]
data["HasGarage"] = (data["GarageArea"] > 0).astype(int)
data["HasBasement"] = (data["TotalBsmtSF"] > 0).astype(int)
data["HasFireplace"] = (data["Fireplaces"] > 0).astype(int)
data["HasPool"] = (data["PoolArea"] > 0).astype(int)
data["TotalPorchSF"] = (
    data["OpenPorchSF"]
    + data["EnclosedPorch"]
    + data["3SsnPorch"]
    + data["ScreenPorch"]
    + data["WoodDeckSF"]
)

print("Engineered features added.")
data[["TotalSF", "TotalBath", "HouseAge", "RemodAge", "TotalPorchSF"]].describe().T

## 4. Train / test split, categorical encoding, and scaling

**Encoding:** The dataset has many categorical columns (`Neighborhood`, `MSZoning`, quality ratings, etc.).
Models require numeric input, so we **label-encode** categoricals. This is simple and works well with tree models.
One-hot encoding is a valid alternative for linear models (avoids implying order), but with high-cardinality `Neighborhood`
it creates many sparse columns; label encoding + regularization / trees is a practical trade-off here.

**Split:** 80/20 hold-out with `random_state=42`. True stratification applies to classification; for continuous `SalePrice`
we rely on a large enough random split (and 5-fold CV later). A price-binned stratified split is a possible enhancement.

**Scaling:** `StandardScaler` is fit on the **training** numeric columns only and applied to linear models
(Ridge / Lasso / ElasticNet / Linear Regression). Tree-based models use unscaled features — splits are scale-invariant.


In [ ]:
# Separate target
y = data["SalePrice"].copy()
X = data.drop(columns=["SalePrice"])

# Log-transform target for training (EDA: reduces right-skew)
y_log = np.log1p(y)

# Identify column types
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric features: {len(num_cols)} | Categorical features: {len(cat_cols)}")
print("Categorical encoding: LabelEncoder (trees handle ordinal codes; linear models use regularization)")

# Label-encode categoricals (fit on full cleaned frame before split for reproducibility in this internship notebook)
X_enc = X.copy()
label_maps = {}
for col in cat_cols:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col].astype(str))
    label_maps[col] = le

# Train-test split (fixed seed for fair model comparison)
X_train, X_test, y_train_log, y_test_log, y_train, y_test = train_test_split(
    X_enc, y_log, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Target: log1p(SalePrice) for fitting | metrics reported on original $ scale")


In [ ]:
# Scale numeric features for LINEAR models only (tree models don't need scaling)
# Fit scaler on TRAIN data to avoid leakage
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
print("StandardScaler applied to numeric columns for linear models (train-fit only).")


## 5. Task 2 — Predictive modeling

We train several models on `log(SalePrice)` and evaluate on the original dollar scale after inverting the transform.

**Approach:**
1. Fit strong baselines with sensible defaults (linear + tree ensembles).
2. Run **GridSearchCV** on key models (Ridge, Random Forest, Gradient Boosting, and XGBoost if available)
   to tune hyperparameters with cross-validation — addressing the need for more rigorous tuning.
3. Compare models on RMSE, MAE, R², RMSLE, and CV stability.
4. Interpret the best tree model via **feature importances** (and coefficients for regularized linear models).


In [ ]:
def evaluate_predictions(y_true, y_pred, model_name="Model"):
    # Compute regression metrics on original SalePrice scale
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # RMSLE on positive prices
    rmsle = np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.clip(y_pred, 1, None))))
    return {
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "RMSLE": rmsle,
    }


def train_and_score(model, Xtr, Xte, name, use_scaled=False):
    # Fit model on log target and return metrics + predictions
    Xt_tr = X_train_scaled if use_scaled else Xtr
    Xt_te = X_test_scaled if use_scaled else Xte
    model.fit(Xt_tr, y_train_log)
    pred_log = model.predict(Xt_te)
    pred = np.expm1(pred_log)
    metrics = evaluate_predictions(y_test, pred, name)
    # 5-fold CV RMSE on log target
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = -cross_val_score(
        model, Xt_tr, y_train_log, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
    )
    metrics["CV_RMSE_log"] = cv_scores.mean()
    metrics["CV_RMSE_log_std"] = cv_scores.std()
    return metrics, pred, model


results = []
predictions = {}
trained_models = {}
print("Evaluation helpers ready (RMSE/MAE/R2/RMSLE + 5-fold CV).")


In [ ]:
# 1) Linear Regression (baseline)
m, pred, fitted = train_and_score(LinearRegression(), X_train, X_test, "Linear Regression", use_scaled=True)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 2) Ridge Regression
m, pred, fitted = train_and_score(Ridge(alpha=10.0, random_state=42), X_train, X_test, "Ridge", use_scaled=True)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 3) Lasso Regression
m, pred, fitted = train_and_score(Lasso(alpha=0.001, max_iter=5000, random_state=42), X_train, X_test, "Lasso", use_scaled=True)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 4) Elastic Net
m, pred, fitted = train_and_score(
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=42),
    X_train, X_test, "ElasticNet", use_scaled=True
)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 5) Decision Tree
m, pred, fitted = train_and_score(
    DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42),
    X_train, X_test, "Decision Tree", use_scaled=False
)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 6) Random Forest
m, pred, fitted = train_and_score(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    X_train, X_test, "Random Forest", use_scaled=False
)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 7) Gradient Boosting
m, pred, fitted = train_and_score(
    GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    X_train, X_test, "Gradient Boosting", use_scaled=False
)
results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
print(m)

In [ ]:
# 8) XGBoost (if available)
if HAS_XGB:
    m, pred, fitted = train_and_score(
        XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror"
        ),
        X_train, X_test, "XGBoost", use_scaled=False
    )
    results.append(m); predictions[m["Model"]] = pred; trained_models[m["Model"]] = fitted
    print(m)
else:
    print("XGBoost not installed — skipped.")

### 5.1 Hyperparameter tuning with GridSearchCV

Default hyperparameters are a useful baseline, but **grid search with cross-validation**
systematically explores a small, high-impact search space and often improves generalization.
We tune:
- **Ridge** (`alpha`) — linear, interpretable baseline
- **Random Forest** (`n_estimators`, `max_depth`, `min_samples_leaf`)
- **Gradient Boosting** (`n_estimators`, `learning_rate`, `max_depth`)
- **XGBoost** (when available) — `n_estimators`, `learning_rate`, `max_depth`

Search grids are intentionally compact so the notebook stays runnable in a reasonable time while still demonstrating proper CV-based tuning.


In [ ]:
# Compact grids — enough to demonstrate CV tuning without extremely long runtimes
cv = KFold(n_splits=3, shuffle=True, random_state=42)
tuned_summaries = []

def run_grid(name, estimator, param_grid, use_scaled=False):
    Xt = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled if use_scaled else X_test
    grid = GridSearchCV(
        estimator,
        param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    grid.fit(Xt, y_train_log)
    best = grid.best_estimator_
    pred = np.expm1(best.predict(Xte))
    metrics = evaluate_predictions(y_test, pred, name)
    # CV score from grid (log-RMSE)
    metrics["CV_RMSE_log"] = -grid.best_score_
    metrics["CV_RMSE_log_std"] = pd.DataFrame(grid.cv_results_).loc[grid.best_index_, "std_test_score"]
    results.append(metrics)
    predictions[name] = pred
    trained_models[name] = best
    tuned_summaries.append({"Model": name, "best_params": grid.best_params_, "best_CV_RMSE_log": -grid.best_score_})
    print(f"{name} best params: {grid.best_params_}")
    print(metrics)
    return best

# Ridge tuning (scaled features)
run_grid(
    "Ridge (GridSearch)",
    Ridge(),
    {"alpha": [0.1, 1.0, 10.0, 30.0, 100.0]},
    use_scaled=True,
)

# Random Forest tuning
run_grid(
    "Random Forest (GridSearch)",
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {
        "n_estimators": [200, 400],
        "max_depth": [12, 20, None],
        "min_samples_leaf": [1, 2],
    },
    use_scaled=False,
)

# Gradient Boosting tuning
run_grid(
    "Gradient Boosting (GridSearch)",
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
    },
    use_scaled=False,
)

if HAS_XGB:
    run_grid(
        "XGBoost (GridSearch)",
        XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror"),
        {
            "n_estimators": [300, 500],
            "learning_rate": [0.03, 0.05],
            "max_depth": [3, 4],
        },
        use_scaled=False,
    )

print("\nTuning summary:")
pd.DataFrame(tuned_summaries)


## 6. Model Comparison Report

We compare **default** models and **GridSearchCV-tuned** variants on the same hold-out test set.
Primary ranking key: **Test RMSE** (lower is better). We also inspect R², MAE, RMSLE, and CV stability.


In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df["RMSE"] = results_df["RMSE"].round(2)
results_df["MAE"] = results_df["MAE"].round(2)
results_df["R2"] = results_df["R2"].round(4)
results_df["RMSLE"] = results_df["RMSLE"].round(4)
results_df["CV_RMSE_log"] = results_df["CV_RMSE_log"].round(4)
results_df["CV_RMSE_log_std"] = results_df["CV_RMSE_log_std"].round(4)

print("Model Comparison (sorted by Test RMSE — lower is better)")
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=results_df, x="RMSE", y="Model", ax=axes[0], color="#2a6f97")
axes[0].set_title("Test RMSE by Model (lower is better)")

sns.barplot(data=results_df, x="R2", y="Model", ax=axes[1], color="#01497c")
axes[1].set_title("Test R² by Model (higher is better)")

plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_pred = predictions[best_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best_pred, alpha=0.5, color="#2a6f97")
mn = min(y_test.min(), best_pred.min())
mx = max(y_test.max(), best_pred.max())
axes[0].plot([mn, mx], [mn, mx], "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual SalePrice")
axes[0].set_ylabel("Predicted SalePrice")
axes[0].set_title(f"{best_name}: Actual vs Predicted")
axes[0].legend()

resid = y_test.values - best_pred
axes[1].hist(resid, bins=30, color="#61a5c2", edgecolor="white")
axes[1].set_title(f"{best_name}: Residual Distribution")
axes[1].set_xlabel("Actual - Predicted")

plt.tight_layout()
plt.show()

print(f"Best model by Test RMSE: {best_name}")
print(results_df.iloc[0].to_dict())

In [ ]:
# --- Model interpretability ---
# Tree ensembles (RF / GB / XGBoost) are strong predictors but less transparent than linear models.
# Feature importances show which inputs drove splits most often / with largest impurity reduction.
# For business communication, pair importances with a regularized linear model (coefficients).

importance_model_name = None
for candidate in [
    "XGBoost (GridSearch)", "Gradient Boosting (GridSearch)", "Random Forest (GridSearch)",
    "XGBoost", "Gradient Boosting", "Random Forest", "Decision Tree"
]:
    if candidate in trained_models and hasattr(trained_models[candidate], "feature_importances_"):
        importance_model_name = candidate
        break

if importance_model_name:
    importances = pd.Series(
        trained_models[importance_model_name].feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False).head(15)

    plt.figure(figsize=(10, 5))
    sns.barplot(x=importances.values, y=importances.index, color="#2a6f97")
    plt.title(f"Top 15 Feature Importances — {importance_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    print(importances)
else:
    print("No tree-based model with feature_importances_ available.")

# Linear interpretability: top |coefficients| from best Ridge variant if available
ridge_name = "Ridge (GridSearch)" if "Ridge (GridSearch)" in trained_models else ("Ridge" if "Ridge" in trained_models else None)
if ridge_name:
    coefs = pd.Series(trained_models[ridge_name].coef_, index=X_train.columns)
    top_coef = coefs.reindex(coefs.abs().sort_values(ascending=False).head(15).index)
    plt.figure(figsize=(10, 5))
    colors = ["#01497c" if v >= 0 else "#9a031e" for v in top_coef.values]
    sns.barplot(x=top_coef.values, y=top_coef.index, palette=colors)
    plt.title(f"Top |Coefficients| — {ridge_name} (log-price scale)")
    plt.xlabel("Coefficient")
    plt.tight_layout()
    plt.show()
    print(top_coef)


### 6.0 Interpretability notes

| Model family | How to explain predictions | Limitation |
|--------------|---------------------------|------------|
| Ridge / Lasso | Signed coefficients on scaled features (direction + relative strength) | Assumes approximate linearity; label-encoded categoricals are not truly ordinal |
| Random Forest | Mean decrease in impurity (feature importance); can add permutation importance | Importances are not causal; correlated features share credit |
| Gradient Boosting / XGBoost | Gain-based feature importance; partial dependence for key drivers | Harder to explain a single house price without tools like SHAP (future work) |

**Practical recommendation:** Use the best boosting/forest model for **accuracy**, and Ridge coefficients + tree importances together when presenting drivers to management (quality, living area, neighborhood, bathrooms, age).


### 6.1 Production recommendation

Pick the model with the **lowest Test RMSE / RMSLE** and stable cross-validation score.
GridSearch-tuned models should be preferred over untuned counterparts when they win on both test and CV metrics.

| Goal | Prefer |
|------|--------|
| Highest accuracy | Tuned Gradient Boosting / XGBoost (usually) |
| Simple baseline / interpretability | Tuned Ridge (coefficients) |
| Fast & robust default | Tuned Random Forest |

**Internship recommendation:** Deploy the best-scoring tuned ensemble for production price estimates.
Keep **Ridge** as an interpretable companion model to explain how quality, size, and age move log-prices — this addresses stakeholder needs for both accuracy and transparency.


## 7. Report on Challenges Faced

| Challenge | Why it mattered | Technique used | Reason |
|-----------|-----------------|----------------|--------|
| High missingness in amenity columns (`PoolQC`, `Alley`, `Fence`, …) | Dropping rows would destroy most of the dataset; filling with mode would invent fake amenities | Fill categorical NA with `"None"` and related numeric fields with `0` | NA usually means the amenity does not exist |
| `LotFrontage` missing (~18%), skewed | Mean imputation would overstate frontage for typical lots | Neighborhood-wise **median**, then global median | Median is robust to large-lot outliers; location-aware |
| Right-skewed `SalePrice` | Hurts linear/squared-error models | Train on `log1p(SalePrice)`, predict then `expm1` | Stabilizes variance and improves RMSLE |
| Extreme outliers (huge `GrLivArea`, low price) | Distort slopes in linear models | Removed a few extreme points after visual EDA | Clean signal for learning |
| Mixed categorical + numeric features | Models need numeric input | Label encoding; scaling for linear models only | Practical for trees; regularization helps linear models |
| Rare / low-count neighborhoods | Local predictions unstable (sparse-category problem) | Keep encoded neighborhood + CV; note as limitation | Similar risk to rare-class imbalance in classification |
| Many collinear size features | Inflates variance of OLS | Ridge / Lasso / ElasticNet + engineered `TotalSF` | Regularization + condensed size signal |
| Untuned hyperparameters | Defaults may underfit/overfit | **GridSearchCV** on Ridge, RF, GB, XGBoost | Evidence-based hyperparameter selection |
| Complex model opacity (RF / boosting) | Stakeholders need explanations | Feature importances + Ridge coefficients | Accuracy + directional business story |
| Choosing one production model | Stakeholders need a clear decision | Compare RMSE, MAE, R², RMSLE + CV stability | Evidence-based selection |

### Extra notes for internship learning
- Always separate **train** and **test** before evaluating — never tune using test scores only (tuning uses CV on train).
- Document cleaning rules so the pipeline is reproducible.
- Connect every major EDA chart to a concrete preprocessing or modeling decision (this notebook’s review focus).


## 8. Final conclusion

1. **EDA → action:** Skewed prices → log target; amenity NA → `"None"`/`0`; skewed `LotFrontage` → **median** (not mean); collinear size → `TotalSF` + regularization; rare neighborhoods → acknowledged sparsity risk.
2. **Preprocessing:** Categoricals were label-encoded; numeric features were standardized **only** for linear models; train/test split was fixed for fair comparison.
3. **Models:** Linear baselines, trees, and boosting were trained on log-price; **GridSearchCV** further tuned Ridge, Random Forest, Gradient Boosting, and XGBoost.
4. **Best model:** The top row in the comparison table (lowest Test RMSE). Prefer the tuned ensemble for forecasts; keep Ridge for explainability.
5. **Business takeaway:** Overall quality, living area / total SF, garage capacity, and neighborhood remain the strongest levers on predicted price.

### Specific future improvements (with expected benefit)

| Improvement | What to do | Expected benefit (order-of-magnitude) |
|-------------|------------|----------------------------------------|
| More / fresher sales data | Collect recent transactions, especially in low-count neighborhoods | Roughly **5–15%** RMSE reduction in sparse areas; more stable local estimates |
| Better categorical encoding | Target / leave-one-out encoding for `Neighborhood`; ordinal maps for quality fields | Often **2–8%** RMSE gain vs label encoding for linear models |
| Richer feature engineering | Interaction terms (quality × area), cluster “neighborhood groups”, remodel indicators | Incremental **1–5%** RMSE gain; clearer product levers for management |
| Advanced stacking / blending | Stack Ridge + RF + GB/XGBoost with a light meta-learner | Typical **1–3%** RMSE improvement when base models err differently |
| Explainability tooling | SHAP values for the production boosting model | Does not reduce RMSE, but improves trust and debugging of individual quotes |
| Price-bin stratified split | Bin `SalePrice` and stratify train/test | More balanced evaluation across cheap vs luxury homes; fairer model selection |

**Limitation to keep visible:** overlapping size signals and sparse neighborhoods mean even a strong model can mis-price unusual homes; monitoring residuals by neighborhood should be part of any deployment.


---
**End of internship notebook** — all required tasks (analysis, prediction, model comparison, challenges) are included in this single file, with EDA findings explicitly tied to preprocessing/modeling choices, CV-based hyperparameter tuning, and interpretability notes.
